# Feature Extraction

Proses ini bertujuan untuk mengonversi teks yang sudah bersih menjadi representasi vektor angka agar dapat diproses oleh algoritma Machine Learning. Teknik yang digunakan meliputi **TF-IDF**, **Word2Vec**, **FastText**, dan **GloVe**.

## 1. Persiapan Data
Memuat dataset yang telah dibersihkan pada tahap preprocessing.

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

# Unduh punkt untuk tokenisasi
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load data
df = pd.read_csv('../Dataset/fake_news_dataset_cleaned.csv')

# Pastikan tidak ada missing values pada teks
docs = df['clean_text'].fillna('').astype(str).tolist()

# Mapping label ke numerik
y = df['label'].map({'real': 0, 'fake': 1}).values

print(f"Total Data: {len(docs)}")


Total Data: 20000


In [2]:
df.head()

,title,text,date,source,author,category,label,text_clean,title_clean,tokens,clean_text,word_count_original,word_count_clean
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real,more tax development both store agreement lawy...,foreign democrat final,"['tax', 'development', 'store', 'agreement', '...",tax development store agreement lawyer hear ou...,216,192
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake,probably guess western behind likely next inve...,to offer down resource great point,"['probably', 'guess', 'western', 'behind', 'li...",probably guess western behind likely next inve...,238,219
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake,them identify forward present success risk sev...,himself church myself carry,"['identify', 'forward', 'present', 'success', ...",identify forward present success risk several ...,222,200
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake,phone which item yard republican safe where po...,you unit its should,"['phone', 'item', 'yard', 'republican', 'safe'...",phone item yard republican safe police identif...,247,228
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake,wonder myself fact difficult course forget exa...,billion believe employee summer how,"['wonder', 'fact', 'difficult', 'course', 'for...",wonder fact difficult course forget exactly pa...,215,195


## 2. Ekstraksi Fitur: TF-IDF
**Term Frequency-Inverse Document Frequency (TF-IDF)** mengukur seberapa penting sebuah kata dalam suatu dokumen terhadap seluruh corpus.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack
import joblib
import os
out_dir = 'extracted_features'
os.makedirs(out_dir, exist_ok=True)
# Menggunakan ngram_range=(1, 3) untuk menangkap frasa kontekstual
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 3))
X_tfidf_text = tfidf.fit_transform(docs)
# One-Hot Encoding untuk fitur source dan category
encoder = OneHotEncoder(handle_unknown='ignore')
X_metadata = encoder.fit_transform(df[['source', 'category']])
# Gabungkan TF-IDF dengan Metadata
X_tfidf = hstack([X_tfidf_text, X_metadata])
vocab_tfidf = tfidf.get_feature_names_out()
print("TF-IDF Text Features Size:", len(vocab_tfidf))
print("Metadata Features Size:", X_metadata.shape[1])
print("Combined TF-IDF + Metadata Matrix Shape:", X_tfidf.shape)
joblib.dump(encoder, f'{out_dir}/metadata_encoder.pkl')


TF-IDF Text Features Size: 5000
Metadata Features Size: 16
Combined TF-IDF + Metadata Matrix Shape: (20000, 5016)


['extracted_features/metadata_encoder.pkl']

## 3. Ekstraksi Fitur: Word Embedding (Word2Vec)
**Word2Vec** mempelajari representasi vektor dari kata berdasarkan konteks kemunculannya (menggunakan metode Skip-gram/CBOW).

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import os
out_dir = 'extracted_features'
os.makedirs(out_dir, exist_ok=True)
max_words = 10000
max_len = 200
# Tokenizer Keras
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(docs)
X_seq = tokenizer.texts_to_sequences(docs)
X_seq_pad = pad_sequences(X_seq, maxlen=max_len, padding='post')
print("Sequence Matrix Shape:", X_seq_pad.shape)
with open(f'{out_dir}/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
np.save(f'{out_dir}/X_seq.npy', X_seq_pad)
word_index = tokenizer.word_index
print(f"Total Unique Tokens: {len(word_index)}")
# Word2Vec Embedding Matrix
from gensim.models import Word2Vec
tokenized_docs = [doc.split() for doc in docs]
w2v_model = Word2Vec(sentences=tokenized_docs, vector_size=100, window=5, min_count=2, sg=1, epochs=10, workers=4)
embedding_matrix_w2v = np.zeros((max_words, 100))
for word, i in word_index.items():
    if i < max_words:
        if word in w2v_model.wv:
            embedding_matrix_w2v[i] = w2v_model.wv[word]
np.save(f'{out_dir}/embedding_matrix_w2v.npy', embedding_matrix_w2v)
print("Word2Vec Embedding Matrix Shape:", embedding_matrix_w2v.shape)


Sequence Matrix Shape: (20000, 200)
Total Unique Tokens: 869


Word2Vec Embedding Matrix Shape: (10000, 100)


## 4. Ekstraksi Fitur: Word Embedding (FastText)
**FastText** mirip dengan Word2Vec namun merepresentasikan kata sebagai kumpulan n-gram karakter, sehingga dapat menangani kata-kata yang tidak ada dalam vocabulary (Out-Of-Vocabulary).

In [5]:
from gensim.models import FastText
import numpy as np
import os
out_dir = 'extracted_features'
ft_model = FastText(sentences=tokenized_docs, vector_size=100, window=5, min_count=2, sg=1, epochs=10, workers=4)
embedding_matrix_ft = np.zeros((max_words, 100))
for word, i in word_index.items():
    if i < max_words:
        if word in ft_model.wv:
            embedding_matrix_ft[i] = ft_model.wv[word]
np.save(f'{out_dir}/embedding_matrix_ft.npy', embedding_matrix_ft)
print("FastText Embedding Matrix Shape:", embedding_matrix_ft.shape)


FastText Embedding Matrix Shape: (10000, 100)


## 5. Ekstraksi Fitur: Word Embedding (GloVe)
**GloVe (Global Vectors for Word Representation)** memanfaatkan probabilitas kemunculan bersama kata dalam corpus secara global. Di sini kita menggunakan pre-trained GloVe model dari library Gensim.

In [6]:
import gensim.downloader as api
import numpy as np
print("Loading pre-trained GloVe model...")
glove_model = api.load('glove-wiki-gigaword-100')
embedding_matrix_glove = np.zeros((max_words, 100))
for word, i in word_index.items():
    if i < max_words:
        if word in glove_model:
            embedding_matrix_glove[i] = glove_model[word]
np.save(f'{out_dir}/embedding_matrix_glove.npy', embedding_matrix_glove)
print("GloVe Embedding Matrix Shape:", embedding_matrix_glove.shape)


Loading pre-trained GloVe model...


GloVe Embedding Matrix Shape: (10000, 100)


## 6. Penyimpanan Hasil Ekstraksi Fitur
Menyimpan matriks fitur dan label ke dalam file untuk digunakan pada tahap Modeling.

In [7]:
from scipy import sparse
import os
import numpy as np
out_dir = 'extracted_features'
os.makedirs(out_dir, exist_ok=True)
sparse.save_npz(f'{out_dir}/X_tfidf.npz', X_tfidf)
np.save(f'{out_dir}/y_labels.npy', y)
print(f"Semua fitur berhasil disimpan di dalam folder '{out_dir}/'")


Semua fitur berhasil disimpan di dalam folder 'extracted_features/'
